In [1]:
config_path = "config/params.yaml"
mode = "default"

In [2]:
# Parameters
config_path = "config/params.yaml"


# 02 — Bangun tabel event

**Tujuan:** mengubah raw menjadi tabel event dan pesan commit yang bebas data pribadi mentah.
Email dibuang saat parse, dan login diganti pseudonim HMAC.

In [3]:
from concurrent.futures import ProcessPoolExecutor

import pandas as pd
import polars as pl

from ghremoval.config import load_context
from ghremoval.parse.events import build_hour
from ghremoval.quality import Checks, pii_columns
from ghremoval.runlog import RunLog
from ghremoval.secrets import load_hmac_key

ctx = load_context(config_path)
run = RunLog("02_build_event_tables", ctx.paths.run_log, ctx.paths.root)
checks = Checks()
key = load_hmac_key(ctx).reveal().encode()
hours = ctx.all_hours()

## 1. Parse per jam

In [4]:
args = [(ctx.paths.raw / f"{hour}.json.gz", ctx.paths.interim, key, hour, week) for hour, week in hours]
with ProcessPoolExecutor(max_workers=4) as pool:
    stats = pd.DataFrame(list(pool.map(build_hour, *zip(*args))))
stats

,hour_id,week,n_events,n_commits,n_actors,n_bad,n_duplicates,n_push_truncated
0,2024-04-01-15,dev,272191,214555,64864,0,0,1076
1,2024-04-09-3,w0,210306,175273,42256,0,0,678
2,2024-04-09-15,w0,280409,240072,71558,0,0,1386
3,2024-04-11-3,w0,208923,179262,41233,0,0,789
4,2024-04-11-15,w0,265679,220473,71798,0,0,1381
5,2024-04-13-3,w0,145220,124908,28500,0,0,542
6,2024-04-13-15,w0,193562,161316,47248,0,0,693
7,2024-04-16-3,w1,176054,145229,43599,0,0,793
8,2024-04-16-15,w1,256809,207208,75179,0,0,1330
9,2024-04-18-3,w1,187898,244118,42367,0,0,5089


## Ringkasan & cek

In [5]:
events = pl.scan_parquet(str(ctx.paths.interim / "events/*/part.parquet"))
per_hour = events.group_by("hour_id").agg(pl.len().alias("n"), pl.col("event_id").n_unique().alias("n_unique"), pl.col("second_of_hour").min().alias("s_min"), pl.col("second_of_hour").max().alias("s_max")).collect()
checks.add("Q04a", (per_hour["n"] == per_hour["n_unique"]).all(), "event_id unik per jam")
checks.add("Q04b", per_hour["s_min"].min() >= 0 and per_hour["s_max"].max() <= 3599, "second_of_hour dalam rentang")
hour_mismatch = events.with_columns(pl.col("created_at").str.slice(0, 13).str.replace("T", "-").alias("stamp"), pl.col("hour_id").str.replace(r"-(\d)$", "-0$1").alias("norm")).filter(pl.col("stamp") != pl.col("norm")).select(pl.len()).collect().item()
checks.add("Q04c", hour_mismatch == 0, f"{hour_mismatch} event di jam yang salah")
columns = events.collect_schema().names() + pl.scan_parquet(str(ctx.paths.interim / "commit_msgs/*/part.parquet")).collect_schema().names()
checks.add("Q05", not pii_columns(columns), f"kolom mencurigakan: {pii_columns(columns)}")
dev = stats[stats["week"] == "dev"].iloc[0]
checks.add("sanity_events", int(dev["n_events"]) == 272191, f"event jam dev {dev['n_events']}")
checks.add("sanity_truncated", int(dev["n_push_truncated"]) == 1076, f"push terpotong {dev['n_push_truncated']}")
checks.raise_if_failed()
print(run.finish(n_events=int(stats["n_events"].sum()), n_commits=int(stats["n_commits"].sum())))

[LULUS] Q04a: event_id unik per jam
[LULUS] Q04b: second_of_hour dalam rentang
[LULUS] Q04c: 0 event di jam yang salah
[LULUS] Q05: kolom mencurigakan: []
[LULUS] sanity_events: event jam dev 272191
[LULUS] sanity_truncated: push terpotong 1076
{'notebook': '02_build_event_tables', 'finished_at_utc': '2026-11-10T05:52:10+00:00', 'git_commit': 'bdf6ff7', 'duration_seconds': 38.3, 'counts': {'n_events': 5291889, 'n_commits': 6041629}}
